###1) Wczytanie i kontrola Gold

In [0]:
from datetime import date

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)
dbutils.widgets.text(
    "train_end_date_local_exclusive",
    "2024-09-01"
)
dbutils.widgets.text(
    "validation_end_date_local_exclusive",
    "2024-10-01"
)
dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

EVALUATION_START_LOCAL = dbutils.widgets.get(
    "evaluation_start_date_local"
)
TRAIN_END_LOCAL_EXCLUSIVE = dbutils.widgets.get(
    "train_end_date_local_exclusive"
)
VALIDATION_END_LOCAL_EXCLUSIVE = dbutils.widgets.get(
    "validation_end_date_local_exclusive"
)
TEST_END_LOCAL_EXCLUSIVE = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

split_boundaries = [
    date.fromisoformat(EVALUATION_START_LOCAL),
    date.fromisoformat(TRAIN_END_LOCAL_EXCLUSIVE),
    date.fromisoformat(VALIDATION_END_LOCAL_EXCLUSIVE),
    date.fromisoformat(TEST_END_LOCAL_EXCLUSIVE),
]

if split_boundaries != sorted(split_boundaries):
    raise ValueError("Granice podziału nie są chronologiczne.")

if len(set(split_boundaries)) != 4:
    raise ValueError("Każdy przedział musi mieć dodatnią długość.")

GOLD_TABLE = (
    "workspace.energy_gold."
    "market_decision_replay_15m"
)

gold_df = spark.table(GOLD_TABLE)

gold_summary_df = (
    gold_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("is_model_ready").cast("int")
        ).alias("model_ready_count"),
        F.sum(
            (~F.col("is_model_ready")).cast("int")
        ).alias("not_model_ready_count"),
        F.min("decision_time_utc").alias(
            "min_decision_time_utc"
        ),
        F.max("decision_time_utc").alias(
            "max_decision_time_utc"
        ),
        F.min("target_interval_start_utc").alias(
            "min_target_time_utc"
        ),
        F.max("target_interval_end_utc").alias(
            "max_target_time_utc"
        )
    )
)

print(f"Tabela: {GOLD_TABLE}")
print(f"Liczba kolumn: {len(gold_df.columns)}")
print(
    "Podział: ",
    EVALUATION_START_LOCAL,
    "—",
    TRAIN_END_LOCAL_EXCLUSIVE,
    "—",
    VALIDATION_END_LOCAL_EXCLUSIVE,
    "—",
    TEST_END_LOCAL_EXCLUSIVE,
)

display(gold_summary_df)

###2) Klasyfikacja kolumn

In [0]:
identifier_columns = [
    "decision_id",
    "system_id",
    "node_id",
    "decision_time_utc",
    "target_interval_start_utc",
    "target_interval_end_utc",
]


feature_columns = [
    "decision_horizon_minutes",

    # Stan farmy znany w chwili decyzji
    "observed_ac_power_kw",
    "observed_pv_energy_mwh",
    "observed_poa_wm2",
    "observed_temperature_c",
    "observed_wind_speed_ms",
    "observed_wind_direction_deg",

    # Historyczna prognoza pogody
    "forecast_temperature_c",
    "forecast_relative_humidity_pct",
    "forecast_cloud_cover_pct",
    "forecast_wind_speed_ms",
    "forecast_wind_direction_deg",
    "forecast_shortwave_radiation_wm2",
    "forecast_gti_wm2",
    "forecast_is_day",
    "weather_forecast_age_minutes",
    "weather_lead_time_hours",

    # Znany rynek dnia następnego
    "dam_lmp_usd_mwh",
    "dam_energy_component_usd_mwh",
    "dam_congestion_component_usd_mwh",
    "dam_loss_component_usd_mwh",
    "dam_ghg_component_usd_mwh",

    # Wyłącznie wcześniejsze ceny FMM
    "last_known_fmm_lmp_usd_mwh",
    "mean_last_4_known_fmm_lmp_usd_mwh",
    "min_last_4_known_fmm_lmp_usd_mwh",
    "max_last_4_known_fmm_lmp_usd_mwh",
    "known_fmm_price_count_last_4",
    "last_known_fmm_price_age_minutes",
    "last_known_fmm_to_target_gap_minutes",
]


label_columns = [
    # Wyniki poznane dopiero po interwale
    "actual_pv_power_kw",
    "actual_pv_energy_mwh",
    "actual_pv_power_is_outlier",
    "actual_fmm_lmp_usd_mwh",
    "actual_fmm_energy_component_usd_mwh",
    "actual_fmm_congestion_component_usd_mwh",
    "actual_fmm_loss_component_usd_mwh",
    "actual_fmm_is_price_spike",
    "actual_fmm_minus_dam_spread_usd_mwh",
]


quality_columns = [
    "is_feature_complete",
    "is_label_complete",
    "is_model_ready",
    "feature_quality_status",
    "label_quality_status",
]


required_columns = (
    identifier_columns
    + feature_columns
    + label_columns
    + quality_columns
)


missing_columns = sorted(
    set(required_columns) - set(gold_df.columns)
)

feature_label_overlap = sorted(
    set(feature_columns) & set(label_columns)
)


print(f"Liczba identyfikatorów: {len(identifier_columns)}")
print(f"Liczba potencjalnych cech: {len(feature_columns)}")
print(f"Liczba etykiet: {len(label_columns)}")
print(f"Liczba kolumn jakości: {len(quality_columns)}")
print(f"Brakujące kolumny: {missing_columns}")
print(
    "Cechy będące jednocześnie etykietami: "
    f"{feature_label_overlap}"
)


if missing_columns:
    raise RuntimeError(
        "W tabeli Gold brakuje wymaganych kolumn."
    )

if feature_label_overlap:
    raise RuntimeError(
        "Wykryto potencjalny data leakage: "
        "kolumna jest cechą i etykietą."
    )


print()
print("Klasyfikacja kolumn została zaliczona.")

###3) Zbiór model_ready

In [0]:
model_ready_df = (
    gold_df
    .filter(F.col("is_model_ready"))
    .orderBy("target_interval_start_utc")
)


model_ready_summary_df = (
    model_ready_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum(
            F.col("actual_pv_power_kw")
            .isNull()
            .cast("int")
        ).alias("missing_pv_label_count"),
        F.sum(
            F.col("actual_fmm_lmp_usd_mwh")
            .isNull()
            .cast("int")
        ).alias("missing_fmm_label_count"),
        F.sum(
            (~F.col("is_feature_complete"))
            .cast("int")
        ).alias("incomplete_feature_count"),
        F.sum(
            (~F.col("is_label_complete"))
            .cast("int")
        ).alias("incomplete_label_count"),
        F.min("target_interval_start_utc").alias(
            "min_target_time_utc"
        ),
        F.max("target_interval_end_utc").alias(
            "max_target_time_utc"
        )
    )
)


display(model_ready_summary_df)


model_ready_test = (
    model_ready_summary_df
    .first()
    .asDict()
)


if model_ready_test["row_count"] <= 0:
    raise RuntimeError(
        "Zbiór model-ready jest pusty."
    )

if (
    model_ready_test["row_count"]
    != model_ready_test["distinct_decision_id_count"]
):
    raise RuntimeError(
        "W zbiorze model-ready są duplikaty decyzji."
    )

if model_ready_test["missing_pv_label_count"] != 0:
    raise RuntimeError(
        "Brakuje etykiet produkcji PV."
    )

if model_ready_test["missing_fmm_label_count"] != 0:
    raise RuntimeError(
        "Brakuje etykiet ceny FMM."
    )

if model_ready_test["incomplete_feature_count"] != 0:
    raise RuntimeError(
        "Zbiór zawiera niekompletne cechy."
    )

if model_ready_test["incomplete_label_count"] != 0:
    raise RuntimeError(
        "Zbiór zawiera niekompletne etykiety."
    )


print("Kontrola zbioru model-ready została zaliczona.")

###4) Profil potencjalnych cech

In [0]:
profile_expressions = []


for column_name in feature_columns:
    numeric_column = F.col(column_name).cast("double")

    is_missing = (
        F.col(column_name).isNull()
        | F.isnan(numeric_column)
    )

    profile_expressions.extend([
        F.sum(
            is_missing.cast("int")
        ).alias(f"{column_name}__missing"),
        F.countDistinct(
            F.col(column_name)
        ).alias(f"{column_name}__distinct"),
        F.min(
            numeric_column
        ).alias(f"{column_name}__minimum"),
        F.max(
            numeric_column
        ).alias(f"{column_name}__maximum"),
    ])


profile_result = (
    model_ready_df
    .agg(*profile_expressions)
    .first()
    .asDict()
)


feature_profile_rows = []


for column_name in feature_columns:
    minimum = profile_result[
        f"{column_name}__minimum"
    ]
    maximum = profile_result[
        f"{column_name}__maximum"
    ]

    feature_profile_rows.append((
        column_name,
        int(
            profile_result[
                f"{column_name}__missing"
            ]
        ),
        int(
            profile_result[
                f"{column_name}__distinct"
            ]
        ),
        None if minimum is None else float(minimum),
        None if maximum is None else float(maximum),
    ))


feature_profile_df = spark.createDataFrame(
    feature_profile_rows,
    """
    feature_name STRING,
    missing_count LONG,
    distinct_value_count LONG,
    minimum DOUBLE,
    maximum DOUBLE
    """
)


display(
    feature_profile_df
    .orderBy("feature_name")
)


print("Cechy wymagające uwagi:")

display(
    feature_profile_df
    .filter(
        (F.col("missing_count") > 0)
        | (F.col("distinct_value_count") <= 1)
    )
    .orderBy("feature_name")
)

###5) Dzienna kompletność i profil danych

In [0]:
MARKET_TIMEZONE = "America/Los_Angeles"


eda_df = (
    gold_df
    .withColumn(
        "target_time_local",
        F.from_utc_timestamp(
            F.col("target_interval_start_utc"),
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "target_date_local",
        F.to_date("target_time_local")
    )
    .withColumn(
        "target_hour_local",
        F.hour("target_time_local")
    )
)


daily_profile_df = (
    eda_df
    .groupBy("target_date_local")
    .agg(
        F.count("*").alias("interval_count"),
        F.sum(
            F.col("is_model_ready").cast("int")
        ).alias("model_ready_count"),
        F.sum(
            (~F.col("is_model_ready")).cast("int")
        ).alias("not_model_ready_count"),
        F.sum("actual_pv_energy_mwh").alias(
            "actual_pv_energy_mwh"
        ),
        F.avg("actual_pv_power_kw").alias(
            "average_pv_power_kw"
        ),
        F.max("actual_pv_power_kw").alias(
            "maximum_pv_power_kw"
        ),
        F.avg("dam_lmp_usd_mwh").alias(
            "average_dam_lmp_usd_mwh"
        ),
        F.avg("actual_fmm_lmp_usd_mwh").alias(
            "average_fmm_lmp_usd_mwh"
        ),
        F.min("actual_fmm_lmp_usd_mwh").alias(
            "minimum_fmm_lmp_usd_mwh"
        ),
        F.max("actual_fmm_lmp_usd_mwh").alias(
            "maximum_fmm_lmp_usd_mwh"
        ),
        F.avg(
            "actual_fmm_minus_dam_spread_usd_mwh"
        ).alias("average_fmm_dam_spread_usd_mwh"),
        F.sum(
            F.col("actual_fmm_is_price_spike")
            .cast("int")
        ).alias("price_spike_count")
    )
    .orderBy("target_date_local")
)


daily_validation_df = (
    daily_profile_df
    .agg(
        F.count("*").alias("day_count"),
        F.sum("interval_count").alias(
            "total_interval_count"
        ),
        F.min("interval_count").alias(
            "minimum_intervals_per_day"
        ),
        F.max("interval_count").alias(
            "maximum_intervals_per_day"
        ),
        F.sum("model_ready_count").alias(
            "total_model_ready_count"
        ),
        F.sum("not_model_ready_count").alias(
            "total_not_model_ready_count"
        )
    )
)


display(daily_validation_df)
display(daily_profile_df)

###6) Profil według godziny lokalnej

In [0]:
model_ready_eda_df = (
    model_ready_df
    .withColumn(
        "target_time_local",
        F.from_utc_timestamp(
            F.col("target_interval_start_utc"),
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "target_date_local",
        F.to_date("target_time_local")
    )
    .withColumn(
        "target_hour_local",
        F.hour("target_time_local")
    )
)


hourly_profile_df = (
    model_ready_eda_df
    .groupBy("target_hour_local")
    .agg(
        F.count("*").alias("row_count"),
        F.avg("observed_ac_power_kw").alias(
            "average_observed_power_kw"
        ),
        F.avg("forecast_gti_wm2").alias(
            "average_forecast_gti_wm2"
        ),
        F.avg("actual_pv_power_kw").alias(
            "average_actual_pv_power_kw"
        ),
        F.avg("dam_lmp_usd_mwh").alias(
            "average_dam_lmp_usd_mwh"
        ),
        F.avg("actual_fmm_lmp_usd_mwh").alias(
            "average_actual_fmm_lmp_usd_mwh"
        ),
        F.avg(
            "actual_fmm_minus_dam_spread_usd_mwh"
        ).alias(
            "average_fmm_dam_spread_usd_mwh"
        ),
        F.sum(
            F.col("actual_fmm_is_price_spike")
            .cast("int")
        ).alias("price_spike_count")
    )
    .orderBy("target_hour_local")
)


display(hourly_profile_df)


hourly_profile_test_df = (
    hourly_profile_df
    .agg(
        F.count("*").alias("hour_count"),
        F.sum("row_count").alias("total_row_count"),
        F.min("target_hour_local").alias(
            "minimum_hour"
        ),
        F.max("target_hour_local").alias(
            "maximum_hour"
        )
    )
)


display(hourly_profile_test_df)

###7) Chronologiczny podział danych
- **TRAIN**: od początku analizy do `train_end_date_local_exclusive`,
- **VALIDATION**: do `validation_end_date_local_exclusive`,
- **TEST**: do `evaluation_end_date_local_exclusive`.

In [0]:
model_dataset_df = (
    model_ready_eda_df
    .withColumn(
        "dataset_split",
        F.when(
            (
                F.col("target_date_local")
                >= F.lit(
                    EVALUATION_START_LOCAL
                ).cast("date")
            )
            &
            (
                F.col("target_date_local")
                < F.lit(
                    TRAIN_END_LOCAL_EXCLUSIVE
                ).cast("date")
            ),
            F.lit("TRAIN")
        )
        .when(
            (
                F.col("target_date_local")
                >= F.lit(
                    TRAIN_END_LOCAL_EXCLUSIVE
                ).cast("date")
            )
            &
            (
                F.col("target_date_local")
                < F.lit(
                    VALIDATION_END_LOCAL_EXCLUSIVE
                ).cast("date")
            ),
            F.lit("VALIDATION")
        )
        .when(
            (
                F.col("target_date_local")
                >= F.lit(
                    VALIDATION_END_LOCAL_EXCLUSIVE
                ).cast("date")
            )
            &
            (
                F.col("target_date_local")
                < F.lit(
                    TEST_END_LOCAL_EXCLUSIVE
                ).cast("date")
            ),
            F.lit("TEST")
        )
        .otherwise(F.lit("OUT_OF_RANGE"))
    )
)


split_summary_df = (
    model_dataset_df
    .groupBy("dataset_split")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.countDistinct("target_date_local").alias(
            "date_count"
        ),
        F.min("target_interval_start_utc").alias(
            "min_target_time_utc"
        ),
        F.max("target_interval_start_utc").alias(
            "max_target_time_utc"
        ),
        F.avg("actual_pv_power_kw").alias(
            "average_pv_power_kw"
        ),
        F.avg("actual_fmm_lmp_usd_mwh").alias(
            "average_fmm_lmp_usd_mwh"
        )
    )
    .withColumn(
        "_split_order",
        F.when(
            F.col("dataset_split") == "TRAIN", 1
        )
        .when(
            F.col("dataset_split") == "VALIDATION", 2
        )
        .when(
            F.col("dataset_split") == "TEST", 3
        )
        .otherwise(4)
    )
    .orderBy("_split_order")
    .drop("_split_order")
)


display(split_summary_df)

###8) Test podziału czasowego

In [0]:
split_rows = [
    row.asDict()
    for row in split_summary_df.collect()
]


split_results = {
    row["dataset_split"]: row
    for row in split_rows
}


expected_splits = {
    "TRAIN",
    "VALIDATION",
    "TEST",
}


if set(split_results) != expected_splits:
    raise RuntimeError(
        "Nie utworzono dokładnie trzech "
        "oczekiwanych części zbioru."
    )


total_split_rows = sum(
    row["row_count"]
    for row in split_rows
)


if total_split_rows != model_ready_test["row_count"]:
    raise RuntimeError(
        "Podział nie zawiera wszystkich "
        "rekordów model-ready."
    )


for split_name in expected_splits:
    split_row = split_results[split_name]

    if split_row["row_count"] <= 0:
        raise RuntimeError(
            f"Zbiór {split_name} jest pusty."
        )

    if (
        split_row["row_count"]
        != split_row[
            "distinct_decision_id_count"
        ]
    ):
        raise RuntimeError(
            f"Zbiór {split_name} zawiera "
            "duplikaty decision_id."
        )


if not (
    split_results["TRAIN"]["max_target_time_utc"]
    <
    split_results["VALIDATION"]["min_target_time_utc"]
    <
    split_results["VALIDATION"]["max_target_time_utc"]
    <
    split_results["TEST"]["min_target_time_utc"]
):
    raise RuntimeError(
        "Zbiory nie są poprawnie "
        "uporządkowane chronologicznie."
    )


print(f"Łączna liczba rekordów: {total_split_rows}")
print("Brak nakładania się okresów.")
print("Chronologiczny podział został zaliczony.")

###9) Feature engineering

In [0]:
import math


constant_feature_columns = [
    "dam_ghg_component_usd_mwh",
    "known_fmm_price_count_last_4",
    "last_known_fmm_price_age_minutes",
]


raw_circular_feature_columns = [
    "observed_wind_direction_deg",
    "forecast_wind_direction_deg",
]


excluded_model_feature_columns = set(
    constant_feature_columns
    + raw_circular_feature_columns
)


base_model_feature_columns = [
    column_name
    for column_name in feature_columns
    if column_name
    not in excluded_model_feature_columns
]


model_features_df = (
    model_dataset_df

    # Czas lokalny interwału docelowego
    .withColumn(
        "target_time_local",
        F.from_utc_timestamp(
            F.col("target_interval_start_utc"),
            MARKET_TIMEZONE
        )
    )
    .withColumn(
        "target_minute_of_day",
        (
            F.hour("target_time_local") * 60
            + F.minute("target_time_local")
        ).cast("double")
    )
    .withColumn(
        "target_day_of_week",
        (
            F.dayofweek("target_time_local") - 1
        ).cast("double")
    )
    .withColumn(
        "target_quarter",
        (
            F.floor(
                F.minute("target_time_local") / 15
            ) + 1
        ).cast("double")
    )

    # Cykliczna reprezentacja pory dnia
    .withColumn(
        "target_time_of_day_sin",
        F.sin(
            F.lit(2.0 * math.pi)
            * F.col("target_minute_of_day")
            / F.lit(1440.0)
        )
    )
    .withColumn(
        "target_time_of_day_cos",
        F.cos(
            F.lit(2.0 * math.pi)
            * F.col("target_minute_of_day")
            / F.lit(1440.0)
        )
    )

    # Cykliczna reprezentacja dnia tygodnia
    .withColumn(
        "target_day_of_week_sin",
        F.sin(
            F.lit(2.0 * math.pi)
            * F.col("target_day_of_week")
            / F.lit(7.0)
        )
    )
    .withColumn(
        "target_day_of_week_cos",
        F.cos(
            F.lit(2.0 * math.pi)
            * F.col("target_day_of_week")
            / F.lit(7.0)
        )
    )

    # Kierunek wiatru jest zmienną kołową
    .withColumn(
        "observed_wind_direction_sin",
        F.sin(
            F.col("observed_wind_direction_deg")
            * F.lit(math.pi / 180.0)
        )
    )
    .withColumn(
        "observed_wind_direction_cos",
        F.cos(
            F.col("observed_wind_direction_deg")
            * F.lit(math.pi / 180.0)
        )
    )
    .withColumn(
        "forecast_wind_direction_sin",
        F.sin(
            F.col("forecast_wind_direction_deg")
            * F.lit(math.pi / 180.0)
        )
    )
    .withColumn(
        "forecast_wind_direction_cos",
        F.cos(
            F.col("forecast_wind_direction_deg")
            * F.lit(math.pi / 180.0)
        )
    )

    # Relacje między znanymi cenami
    .withColumn(
        "last_known_fmm_minus_dam_usd_mwh",
        F.col("last_known_fmm_lmp_usd_mwh")
        - F.col("dam_lmp_usd_mwh")
    )
    .withColumn(
        "mean_last_4_fmm_minus_dam_usd_mwh",
        F.col("mean_last_4_known_fmm_lmp_usd_mwh")
        - F.col("dam_lmp_usd_mwh")
    )
    .withColumn(
        "last_4_known_fmm_range_usd_mwh",
        F.col("max_last_4_known_fmm_lmp_usd_mwh")
        - F.col("min_last_4_known_fmm_lmp_usd_mwh")
    )
)


engineered_model_feature_columns = [
    "target_quarter",
    "target_time_of_day_sin",
    "target_time_of_day_cos",
    "target_day_of_week_sin",
    "target_day_of_week_cos",
    "observed_wind_direction_sin",
    "observed_wind_direction_cos",
    "forecast_wind_direction_sin",
    "forecast_wind_direction_cos",
    "last_known_fmm_minus_dam_usd_mwh",
    "mean_last_4_fmm_minus_dam_usd_mwh",
    "last_4_known_fmm_range_usd_mwh",
]


final_model_feature_columns = (
    base_model_feature_columns
    + engineered_model_feature_columns
)


print(
    "Wykluczone cechy stałe: "
    f"{constant_feature_columns}"
)

print(
    "Zastąpione cechy kierunkowe: "
    f"{raw_circular_feature_columns}"
)

print(
    "Liczba ostatecznych cech modelu: "
    f"{len(final_model_feature_columns)}"
)


display(
    model_features_df
    .select(
        "target_time_local",
        "dataset_split",
        "target_quarter",
        "target_time_of_day_sin",
        "target_time_of_day_cos",
        "observed_wind_direction_sin",
        "observed_wind_direction_cos",
        "last_known_fmm_lmp_usd_mwh",
        "dam_lmp_usd_mwh",
        "last_known_fmm_minus_dam_usd_mwh",
        "actual_pv_power_kw",
        "actual_fmm_lmp_usd_mwh"
    )
    .orderBy("target_interval_start_utc")
    .limit(12)
)

###10) Walidacja cech modelowych

In [0]:
from pyspark.sql import functions as F

missing_feature_expression = F.lit(0)


for column_name in final_model_feature_columns:
    numeric_column = F.col(column_name).cast("double")

    missing_feature_expression = (
        missing_feature_expression
        +
        (
            F.col(column_name).isNull()
            | F.isnan(numeric_column)
        ).cast("int")
    )


validated_model_features_df = (
    model_features_df
    .withColumn(
        "_missing_feature_count",
        missing_feature_expression
    )
)


feature_validation_df = (
    validated_model_features_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.sum("_missing_feature_count").alias(
            "missing_feature_value_count"
        ),
        F.sum(
            (
                F.col("_missing_feature_count") > 0
            ).cast("int")
        ).alias("rows_with_missing_features"),
        F.sum(
            (
                ~F.col("dataset_split").isin(
                    "TRAIN",
                    "VALIDATION",
                    "TEST"
                )
            ).cast("int")
        ).alias("invalid_split_count"),
        F.min("target_interval_start_utc").alias(
            "min_target_time_utc"
        ),
        F.max("target_interval_end_utc").alias(
            "max_target_time_utc"
        )
    )
)


display(feature_validation_df)


validation_result = (
    feature_validation_df
    .first()
    .asDict()
)


final_feature_label_overlap = sorted(
    set(final_model_feature_columns)
    & set(label_columns)
)

constant_feature_leak = sorted(
    set(final_model_feature_columns)
    & set(constant_feature_columns)
)

raw_direction_feature_leak = sorted(
    set(final_model_feature_columns)
    & set(raw_circular_feature_columns)
)


print(
    "Cechy będące etykietami: "
    f"{final_feature_label_overlap}"
)

print(
    "Cechy stałe pozostawione w modelu: "
    f"{constant_feature_leak}"
)

print(
    "Surowe kierunki pozostawione w modelu: "
    f"{raw_direction_feature_leak}"
)


if validation_result["row_count"] != model_ready_test["row_count"]:
    raise RuntimeError(
        "Niepoprawna liczba rekordów."
    )

if (
    validation_result["row_count"]
    != validation_result[
        "distinct_decision_id_count"
    ]
):
    raise RuntimeError(
        "Wykryto duplikaty decision_id."
    )

if validation_result[
    "missing_feature_value_count"
] != 0:
    raise RuntimeError(
        "W cechach występują braki lub NaN."
    )

if validation_result[
    "rows_with_missing_features"
] != 0:
    raise RuntimeError(
        "Niektóre rekordy mają niepełne cechy."
    )

if validation_result["invalid_split_count"] != 0:
    raise RuntimeError(
        "Występują niepoprawne wartości split."
    )

if final_feature_label_overlap:
    raise RuntimeError(
        "Wykryto data leakage etykiet."
    )

if constant_feature_leak:
    raise RuntimeError(
        "W modelu pozostały cechy stałe."
    )

if raw_direction_feature_leak:
    raise RuntimeError(
        "W modelu pozostały surowe kierunki."
    )


print()
print(
    "Wszystkie testy cech modelowych "
    "zostały zaliczone."
)

###11) Zapis do zbioru modelowego

In [0]:
MODEL_DATASET_TABLE = (
    "workspace.energy_gold."
    "market_model_dataset_15m"
)


model_dataset_output_df = (
    validated_model_features_df
    .drop("_missing_feature_count")
    .select(
        *identifier_columns,
        "target_time_local",
        "target_date_local",
        "dataset_split",
        *final_model_feature_columns,
        *label_columns,
        *quality_columns
    )
    .withColumn(
        "_feature_set_version",
        F.lit("feature_set_v1")
    )
    .withColumn(
        "_source_gold_table",
        F.lit(GOLD_TABLE)
    )
    .withColumn(
        "_feature_created_at_utc",
        F.current_timestamp()
    )
)


(
    model_dataset_output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MODEL_DATASET_TABLE)
)


saved_model_dataset_df = spark.table(
    MODEL_DATASET_TABLE
)


print(f"Zapisano tabelę: {MODEL_DATASET_TABLE}")
print(
    "Liczba rekordów: "
    f"{saved_model_dataset_df.count()}"
)


display(
    saved_model_dataset_df
    .groupBy("dataset_split")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("decision_id").alias(
            "distinct_decision_id_count"
        ),
        F.min("target_interval_start_utc").alias(
            "min_target_time_utc"
        ),
        F.max("target_interval_start_utc").alias(
            "max_target_time_utc"
        )
    )
    .withColumn(
        "_split_order",
        F.when(
            F.col("dataset_split") == "TRAIN", 1
        )
        .when(
            F.col("dataset_split") == "VALIDATION", 2
        )
        .when(
            F.col("dataset_split") == "TEST", 3
        )
        .otherwise(4)
    )
    .orderBy("_split_order")
    .drop("_split_order")
)